In [ ]:
pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 48.6 MB/s  0:00:00


In [ ]:
import pandas as pd
import numpy as np
from gensim.models import Word2Vec
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import string


In [ ]:
df=pd.read_csv("/content/spam.csv", encoding='latin-1')

In [ ]:
df.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   v1          5572 non-null   object
 1   v2          5572 non-null   object
 2   Unnamed: 2  50 non-null     object
 3   Unnamed: 3  12 non-null     object
 4   Unnamed: 4  6 non-null      object
dtypes: object(5)
memory usage: 217.8+ KB


In [ ]:
df.shape

(5572, 5)

In [ ]:
df=df[['v1','v2']]


In [ ]:
df.head()

,v1,v2
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [ ]:
df=df.rename(columns={'v1':'label','v2':'message'})

In [ ]:
df.head()

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [ ]:
df.isnull().sum()

,0
label,0
message,0


In [ ]:
df=df.drop_duplicates()

In [ ]:
import re
import nltk

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

def clean(doc):
  regex='[^a-zA-Z.]'
  doc=re.sub(regex," ",doc)
  doc=doc.lower()
  tokens=nltk.word_tokenize(doc)
  stop_words=set(stopwords.words('english'))
  filtered_tokens=[word for word in tokens if word not in stop_words]
  lemmatizer=WordNetLemmatizer()
  lemmatized_tokens=[lemmatizer.lemmatize(token) for token in filtered_tokens]
  return " ".join(lemmatized_tokens)


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


In [ ]:
df['clean_text']=df['message'].apply(lambda x:clean(x))

In [ ]:
X = df['clean_text'].apply(lambda x: x.split())
y = df['label']

In [ ]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

In [ ]:
print(X_train.shape)
print(X_test.shape)
print(y_test.shape)
print(y_train.shape)
X_train.head()

(4135,)
(1034,)
(1034,)
(4135,)


,clean_text
2228,"[exact, intention]"
5529,"[one, .]"
2149,"[waaaat, lololo, ok, next, time]"
5058,"[free, video, camera, phone, half, price, line..."
5051,"[tick, tick, tick, ...., could, die, lonelines..."


In [ ]:
print(X_train.iloc[0])
print(type(X_train.iloc[0]))

['exact', 'intention']
<class 'list'>


In [ ]:
#Train Word2vec model
word2vec_model=Word2Vec(
    sentences=X_train.tolist(),
    vector_size=100,
    window=5,
    min_count=1,
    workers=4,
    sg=1,
    epochs=10
)

In [ ]:
print(list(word2vec_model.wv.key_to_index.keys())[:30])

['.', 'u', '...', '..', 'call', 'get', 'ur', 'gt', 'lt', 'go', 'know', 'day', 'got', 'like', 'time', 'free', 'come', 'ok', 'good', 'want', 'text', 'need', 'one', 'love', 'r', 'home', 'see', 'p', 'today', 'send']


In [ ]:
print(X_train.iloc[5])
print(type(X_train.iloc[5]))

['tone', 'club', 'sub', 'expired', 'sub', 'reply', 'monoc', 'mono', 'polyc', 'polys', 'weekly', 'p', 'per', 'week', 'txt', 'stop', 'stop', 'msg', 'free', 'stream']
<class 'list'>


In [ ]:
def get_sentence_vector(words):
    vectors = [
        word2vec_model.wv[word]
        for word in words
        if word in word2vec_model.wv
    ]

    if len(vectors) == 0:
        return np.zeros(100)

    return np.mean(vectors, axis=0)

In [ ]:
X_train_vec = np.array(
    X_train.apply(get_sentence_vector).tolist()
)

X_test_vec = np.array(
    X_test.apply(get_sentence_vector).tolist()
)

print("Training shape:", X_train_vec.shape)
print("Testing shape:", X_test_vec.shape)

Training shape: (4135, 100)
Testing shape: (1034, 100)


In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

y_train_num = le.fit_transform(y_train)
y_test_num = le.transform(y_test)

print("Classes:", le.classes_)
print("Training labels:", y_train_num[:10])
print("Testing labels:", y_test_num[:10])

Classes: ['ham' 'spam']
Training labels: [0 0 0 1 0 1 0 0 0 0]
Testing labels: [0 0 0 0 0 1 0 1 0 0]


In [ ]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [optuna]


In [ ]:
import optuna
import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam, RMSprop
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.model_selection import train_test_split

In [ ]:
X_train_new, X_val, y_train_new, y_val = train_test_split(
    X_train_vec,
    y_train_num,
    test_size=0.2,
    random_state=42,
    stratify=y_train_num
)

In [ ]:
def objective(trial):
    tf.keras.backend.clear_session()
    model = Sequential()
    n_layers = trial.suggest_int('n_layers', 1, 3)

    for i in range(n_layers):
        units = trial.suggest_categorical(
            f'units_{i}', [32, 64, 128, 256]
        )
        dropout = trial.suggest_float(
            f'dropout_{i}', 0.1, 0.5
        )

        if i == 0:
            model.add(Dense(units, activation='relu', input_shape=(100,)))
        else:
            model.add(Dense(units, activation='relu'))

        model.add(Dropout(dropout))

    model.add(Dense(1, activation='sigmoid'))
    learning_rate = trial.suggest_float(
        'learning_rate', 1e-4, 1e-2, log=True
    )
    optimizer_name = trial.suggest_categorical(
        'optimizer', ['adam', 'rmsprop']
    )
    batch_size = trial.suggest_categorical(
        'batch_size', [16, 32, 64, 128]
    )

    if optimizer_name == 'adam':
        optimizer = Adam(learning_rate=learning_rate)
    else:
        optimizer = RMSprop(learning_rate=learning_rate)

    model.compile(
        optimizer=optimizer,
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    early_stop = EarlyStopping(
        monitor='val_loss',
        patience=3,
        restore_best_weights=True
    )

    history = model.fit(
        X_train_new,
        y_train_new,
        validation_data=(X_val, y_val),
        epochs=30,
        batch_size=batch_size,
        callbacks=[early_stop],
        verbose=0
    )

    return max(history.history['val_accuracy'])

In [ ]:
study = optuna.create_study(direction='maximize')

study.optimize(objective, n_trials=20)

[I 2026-11-10 09:58:04,016] A new study created in memory with name: no-name-0a510f93-f624-498c-aafa-52bcfc7cd468
/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
[I 2026-11-10 09:58:09,725] Trial 0 finished with value: 0.9613059163093567 and parameters: {'n_layers': 2, 'units_0': 256, 'dropout_0': 0.23489822633641333, 'units_1': 32, 'dropout_1': 0.27599178638071387, 'learning_rate': 0.006874683948787704, 'optimizer': 'adam', 'batch_size': 32}. Best is trial 0 with value: 0.9613059163093567.
[I 2026-11-10 09:58:13,602] Trial 1 finished with value: 0.9613059163093567 and parameters: {'n_layers': 2, 'units_0': 32, 'dropout_0': 0.3555889936940333, 'units_1': 32, 'dropout_1': 0.43470359923724955, 'learning_rate'

In [ ]:
print("Best Validation Accuracy:", study.best_value)
print("Best Hyperparameters:")
print(study.best_params)

Best Validation Accuracy: 0.9721886515617371
Best Hyperparameters:
{'n_layers': 3, 'units_0': 256, 'dropout_0': 0.12328972674620559, 'units_1': 256, 'dropout_1': 0.22903415443204683, 'units_2': 128, 'dropout_2': 0.11287120233475456, 'learning_rate': 0.0006353766629906238, 'optimizer': 'adam', 'batch_size': 64}


In [ ]:
best_params = study.best_params

tf.keras.backend.clear_session()

final_model = Sequential()

for i in range(best_params['n_layers']):

    units = best_params[f'units_{i}']
    dropout = best_params[f'dropout_{i}']

    if i == 0:
        final_model.add(
            tf.keras.Input(shape=(100,))
        )

    final_model.add(Dense(units, activation='relu'))
    final_model.add(Dropout(dropout))

final_model.add(Dense(1, activation='sigmoid'))

final_model.compile(
    optimizer=Adam(
        learning_rate=best_params['learning_rate']
    ),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

final_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 256)            │        25,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 256)            │        65,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 124,673 (487.00 KB)

 Trainable params: 124,673 (487.00 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

history = final_model.fit(
    X_train_new,
    y_train_new,
    validation_data=(X_val, y_val),
    epochs=30,
    batch_size=best_params['batch_size'],
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/30
52/52 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - accuracy: 0.9120 - loss: 0.2517 - val_accuracy: 0.9541 - val_loss: 0.1196
Epoch 2/30
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.9661 - loss: 0.1052 - val_accuracy: 0.9541 - val_loss: 0.1140
Epoch 3/30
52/52 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.9640 - loss: 0.1105 - val_accuracy: 0.9565 - val_loss: 0.1153
Epoch 4/30
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.9661 - loss: 0.0996 - val_accuracy: 0.9613 - val_loss: 0.1180
Epoch 5/30
52/52 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.9674 - loss: 0.0967 - val_accuracy: 0.9589 - val_loss: 0.1070
Epoch 6/30
52/52 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.9689 - loss: 0.0906 - val_accuracy: 0.9589 - val_loss: 0.1177
Epoch 7/30
52/52 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.9686 - loss: 0.0912 - val_accuracy: 0.9613 - val_loss: 0.1027
Epoch 8/30
52/52 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.9713 - loss: 0.0879 - val_accuracy: 0.9589 - val_

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

# Test accuracy
print("Test Accuracy:", final_model.evaluate(X_test_vec, y_test_num)[1])

# Predictions
y_pred = (final_model.predict(X_test_vec) > 0.5).astype(int)

# Evaluation metrics
print(classification_report(y_test_num, y_pred, target_names=['Ham', 'Spam']))

print("Confusion Matrix:\n", confusion_matrix(y_test_num, y_pred))

33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9671 - loss: 0.0849
Test Accuracy: 0.967117965221405
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
              precision    recall  f1-score   support

         Ham       0.97      0.99      0.98       889
        Spam       0.92      0.83      0.88       145

    accuracy                           0.97      1034
   macro avg       0.95      0.91      0.93      1034
weighted avg       0.97      0.97      0.97      1034

Confusion Matrix:
 [[879  10]
 [ 24 121]]


In [ ]:
import joblib

word2vec_model.save("word2vec_model.model")

final_model.save("spam_ann_model.keras")

joblib.dump(le, "label_encoder.pkl")

['label_encoder.pkl']

In [ ]:
from gensim.models import Word2Vec
from tensorflow.keras.models import load_model
import joblib

word2vec_model = Word2Vec.load("word2vec_model.model")
final_model = load_model("spam_ann_model.keras")
le = joblib.load("label_encoder.pkl")

print("All models loaded successfully!")

All models loaded successfully!


In [ ]:
import zipfile
from google.colab import files

with zipfile.ZipFile("spam_models.zip", "w") as z:
    z.write("word2vec_model.model")
    z.write("spam_ann_model.keras")
    z.write("label_encoder.pkl")

files.download("spam_models.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>